# Aula 16 — Ridge na prática

**Aluno · prática guiada e laboratório · cerca de 100 minutos**

Retomamos os **201 automóveis** das aulas de regressão e validação cruzada. A pergunta é: **a penalização do Ridge melhora as previsões de preço?** Vamos escolher α só com os 160 carros de treino. Os 41 carros de teste já foram examinados na Aula 14 e **não serão usados nesta aula**.

Ao terminar, você deverá saber montar `StandardScaler → Ridge` dentro de um `Pipeline`, comparar 13 valores de α nas mesmas 5 partes, interpretar média/desvio e comparar o Ridge com a regressão linear parte a parte.

## Como trabalhar

Execute as células em ordem com **Shift + Enter**. Complete os `TODO` e escreva as respostas em Markdown antes de consultar resultados posteriores. O CSV deve estar em `dados/` ou junto ao notebook; no Colab, envie também o arquivo `automoveis_preparados.csv`.

| Tempo | Etapa |
|---:|---|
| 0–10 min | Retomar a divisão treino/teste e os 5 folds |
| 10–30 min | Ajustar o primeiro Ridge e entender o `Pipeline` |
| 30–55 min | Testar a grade de α e conferir com `GridSearchCV` |
| 55–75 min | Regularizar o polinômio de grau 8 |
| 75–90 min | Comparar erros e coeficientes da regressão múltipla |
| 90–100 min | Escrever a conclusão; desafio opcional com 16 variáveis |

## 1. Ambiente e mesma divisão da Aula 14

`price` é o preço em dólares. Na comparação principal, usaremos quatro entradas: `horsepower`, `curb-weight`, `engine-size` e `highway-mpg`. O arquivo já foi preparado nos encontros anteriores. Não use colunas derivadas de `price` como entrada.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

pd.set_option("display.float_format", lambda valor: f"{valor:,.6g}")
COLUNAS_MULTIPLAS = ["horsepower", "curb-weight", "engine-size", "highway-mpg"]
COLUNAS_16 = [
    "symboling", "normalized-losses", "wheel-base", "length", "width", "height",
    "curb-weight", "engine-size", "bore", "stroke", "compression-ratio",
    "horsepower", "peak-rpm", "city-mpg", "highway-mpg", "fuel_diesel",
]
METRICA = "neg_mean_absolute_error"

In [ ]:
caminhos = [
    Path("dados/automoveis_preparados.csv"),
    Path("aulas/016/dados/automoveis_preparados.csv"),
    Path("automoveis_preparados.csv"),
]
caminho_dados = next((p for p in caminhos if p.is_file()), None)
if caminho_dados is None:
    raise FileNotFoundError("Envie automoveis_preparados.csv para dados/ ou para a pasta do notebook.")

df = pd.read_csv(caminho_dados)
assert len(df) == 201, "Esta prática usa a base preparada de 201 carros."
assert df[COLUNAS_16 + ["price"]].notna().all().all(), "Há ausências nas variáveis da prática."
print(f"Arquivo: {caminho_dados} | {len(df)} carros")
df[COLUNAS_MULTIPLAS + ["price"]].head()

In [ ]:
X = df[COLUNAS_16]
y = df["price"]
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=42
)
X_treino_m = X_treino[COLUNAS_MULTIPLAS]
X_treino_s = X_treino[["horsepower"]]
kf = KFold(n_splits=5, shuffle=True, random_state=42)
alphas = np.logspace(-3, 3, 13)
assert (len(X_treino), len(X_teste)) == (160, 41)
print(f"Treino: {len(X_treino)} | teste guardado: {len(X_teste)}")
print("Grade de α:", np.round(alphas, 4))

> **Antes de continuar:** cada fold usa quantos carros para ajustar e quantos para validar? Por que não podemos escolher α com os 41 carros de teste?

**Sua resposta:** …

## 2. Ponto de partida: regressão linear e um Ridge

O scikit-learn retorna o **MAE negativo** em `cross_val_score`; multiplicamos por −1 para obter erros positivos. Os dois modelos usarão os **mesmos folds**. No Ridge, o `StandardScaler` fica *dentro* do `Pipeline`: em cada rodada ele aprende média e desvio apenas nos 128 carros de ajuste.

In [ ]:
modelo_linear_m = Pipeline([
    ("escala", StandardScaler()),
    ("regressao", LinearRegression()),
])
mae_linear_m = -cross_val_score(
    modelo_linear_m, X_treino_m, y_treino, cv=kf, scoring=METRICA
)
print("LinearRegression | MAE por fold:", np.round(mae_linear_m, 0))
print(f"Média ± desvio: US$ {mae_linear_m.mean():,.0f} ± {mae_linear_m.std():,.0f}")

**Tarefa 1.** Monte `Pipeline([("escala", StandardScaler()), ("ridge", Ridge(alpha=10))])` e calcule os cinco MAEs. O nome do passo `"ridge"` será usado depois em `ridge__alpha`.

In [ ]:
# TODO 1 — complete o Pipeline e a validação cruzada.
# modelo_ridge_10 = Pipeline([...])
# mae_ridge_10 = -cross_val_score(..., X_treino_m, y_treino, cv=kf, scoring=METRICA)
# print(np.round(mae_ridge_10, 0), mae_ridge_10.mean(), mae_ridge_10.std())

> **Explique:** por que `StandardScaler().fit_transform(X_treino_m)` antes de `cross_val_score` causaria vazamento? Por que os coeficientes precisam estar na mesma escala para a penalidade do Ridge?

**Sua resposta:** …

## 3. Escolha de α por validação cruzada

**Tarefa 2.** Para cada um dos 13 valores de `alphas`, monte um novo `Pipeline`, calcule os 5 MAEs e registre a **média**, o **desvio (`ddof=0`)** e as **cinco notas**. Escolha o menor MAE médio. O treino nunca escolhe α: seu erro costuma favorecer α próximo de zero.

In [ ]:
# TODO 2 — complete o laço. Guarde as cinco notas para comparar parte a parte.
# linhas_grade_m = []
# notas_grade_m = {}
# for alpha in alphas:
#     modelo = Pipeline([...])
#     notas = -cross_val_score(..., X_treino_m, y_treino, cv=kf, scoring=METRICA)
#     notas_grade_m[alpha] = notas
#     linhas_grade_m.append({"alpha": alpha, "MAE médio": ..., "desvio": ...})
# grade_m = pd.DataFrame(linhas_grade_m).set_index("alpha")
# alpha_escolhido_m = grade_m["MAE médio"].idxmin()
# display(grade_m)

**Tarefa 3.** Repita a busca com `GridSearchCV`. Passe `{"ridge__alpha": alphas}` ao modelo com passos `"escala"` e `"ridge"`. Confira se `best_params_` e `-best_score_` concordam com sua tabela. O `best_score_` ainda tem sinal negativo.

In [ ]:
# TODO 3 — crie e ajuste a busca usando SOMENTE X_treino_m e y_treino.
# busca_m = GridSearchCV(..., {"ridge__alpha": alphas}, cv=kf, scoring=METRICA)
# busca_m.fit(X_treino_m, y_treino)
# print(busca_m.best_params_, -busca_m.best_score_)

In [ ]:
# Execute após a Tarefa 2. Os dois eixos mostram α em escala logarítmica.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grade_m.index, grade_m["MAE médio"], "o-", color="#108A78", label="Ridge · 4 variáveis")
ax.axhline(mae_linear_m.mean(), color="#D05D52", linestyle="--", label="LinearRegression")
ax.set(xscale="log", xlabel="α (escala log)", ylabel="MAE médio de validação (US$)",
       title="Escolha de α no treino: regressão múltipla")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

> **Registre:** qual α teve o menor MAE médio? Ele está numa ponta da grade? A melhora sobre a LinearRegression parece grande frente ao desvio entre os folds? Se o mínimo estivesse na ponta, o que você faria?

**Sua resposta:** …

## 4. O caso difícil: polinômio de grau 8

Na Aula 14, o grau 8 sobre `horsepower` teve erro de treino pequeno e validação enorme. Primeiro, reproduza essa referência. Depois, use **duas escalas** no Ridge: uma antes do polinômio e outra depois, para colocar `z`, `z²`, …, `z⁸` em unidades comparáveis. Todas as transformações ficam dentro do `Pipeline`.

In [ ]:
modelo_g8_linear = Pipeline([
    ("escala", StandardScaler()),
    ("polinomio", PolynomialFeatures(degree=8, include_bias=False)),
    ("regressao", LinearRegression()),
])
mae_g8_linear = -cross_val_score(
    modelo_g8_linear, X_treino_s, y_treino, cv=kf, scoring=METRICA
)
print("Grau 8 sem Ridge | MAE por fold:", np.round(mae_g8_linear, 0))
print(f"Média ± desvio: US$ {mae_g8_linear.mean():,.0f} ± {mae_g8_linear.std():,.0f}")

**Tarefa 4.** Repita a grade de α para o grau 8. Use `Pipeline`: `StandardScaler → PolynomialFeatures(8, include_bias=False) → StandardScaler → Ridge`. Registre média, desvio e notas. Compare o melhor Ridge de grau 8 com a reta (grau 1 da Aula 14: MAE ≈ US$ 2.891).

In [ ]:
# TODO 4 — busque α para o grau 8 com a segunda escala dos termos polinomiais.
# linhas_grade_g8 = []
# notas_grade_g8 = {}
# for alpha in alphas:
#     modelo = Pipeline([...])
#     notas = -cross_val_score(..., X_treino_s, y_treino, cv=kf, scoring=METRICA)
#     notas_grade_g8[alpha] = notas
#     linhas_grade_g8.append({"alpha": alpha, "MAE médio": ..., "desvio": ...})
# grade_g8 = pd.DataFrame(linhas_grade_g8).set_index("alpha")
# alpha_escolhido_g8 = grade_g8["MAE médio"].idxmin()
# print(grade_g8)

In [ ]:
# Execute após a Tarefa 4. A escala logarítmica permite ver o MAE extremo sem esconder os demais.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grade_g8.index, grade_g8["MAE médio"], "o-", color="#108A78", label="Grau 8 · Ridge")
ax.axhline(mae_g8_linear.mean(), color="#D05D52", linestyle="--", label="Grau 8 · LinearRegression")
ax.axhline(2891.39, color="#8066C2", linestyle=":", label="Grau 1 · Aula 14")
ax.set(xscale="log", yscale="log", xlabel="α (escala log)",
       ylabel="MAE médio de validação (US$, escala log)", title="Ridge controla o grau 8")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

> **Explique:** quanto o MAE médio e o desvio caíram? Mesmo regularizado, o grau 8 venceu a reta? Por que a segunda escala é necessária? O Ridge resolve a extrapolação para potências fora da faixa vista no ajuste?

**Sua resposta:** …

## 5. O Ridge ajudou a regressão múltipla?

**Tarefa 5.** Compare os cinco erros da LinearRegression com os do Ridge escolhido, **na mesma ordem dos folds**. Calcule `LinearRegression − Ridge`: valor positivo favorece o Ridge. Conte em quantos folds ele venceu. Depois ajuste ambos com os **160 carros de treino** para comparar os coeficientes padronizados e a soma de seus quadrados. Isso é uma inspeção dos modelos, **não uma avaliação no teste**.

In [ ]:
# TODO 5 — compare os erros pareados e os coeficientes padronizados.
# mae_ridge_m = notas_grade_m[alpha_escolhido_m]
# comparacao_folds = pd.DataFrame({...})
# comparacao_folds["ganho_Ridge"] = ...
# print(comparacao_folds)
# print("Vitórias do Ridge:", ...)
# modelo_linear_m.fit(X_treino_m, y_treino)
# modelo_ridge_m = Pipeline([... Ridge(alpha=alpha_escolhido_m) ...])
# modelo_ridge_m.fit(X_treino_m, y_treino)
# coeficientes = pd.DataFrame({...}, index=COLUNAS_MULTIPLAS)
# print(coeficientes)

> **Conclua:** um ganho médio de aproximadamente US$ 30, que muda de sinal entre os folds, prova que o Ridge é melhor na regressão múltipla? Cada coeficiente individual precisa diminuir? Por que a **soma dos quadrados** é mais apropriada para entender a penalidade?

**Sua resposta:** …

## 6. Relatório curto

Escreva **4–6 frases**: (1) como escolheu α; (2) MAE médio ± desvio de LinearRegression e Ridge com quatro variáveis; (3) quantos folds o Ridge venceu; (4) o que ocorreu com o grau 8; (5) uma decisão cautelosa para a regressão múltipla. **Não relate MAE de teste nesta aula.**

**Seu relatório:** …

## Desafio opcional — 16 variáveis numéricas

Repita `LinearRegression × Ridge` com `X_treino[COLUNAS_16]`. Use o mesmo `kf` e a mesma grade. Compare as médias e os desvios. **Somente** esta lista de 16 colunas entra no desafio; `price`, `faixa_preco` e colunas derivadas do preço não entram. Se concluir que o Ridge ajuda, lembre que a escolha e a avaliação pelos mesmos folds podem ser um pouco otimistas.

In [ ]:
# DESAFIO (opcional) — avalie as 16 variáveis sem abrir X_teste ou y_teste.
# mae_linear_16 = -cross_val_score(modelo_linear_m, X_treino[COLUNAS_16], y_treino,
#                                  cv=kf, scoring=METRICA)
# ... repita a grade com Pipeline([("escala", StandardScaler()), ("ridge", Ridge(alpha=alpha))])

### Entrega

Entregue o notebook com os **cinco `TODO`** completos, respostas às perguntas e relatório. Confira: mesma divisão e mesmos folds; MAE com sinal corrigido; escala dentro do `Pipeline`; segunda escala no grau 8; teste guardado. O desafio de 16 variáveis é opcional.